# Parte 1: Tarefa oficial

1. Reaproveitar o corpus de 8 documentos da Aula 01.
2. Implementar a matriz TF-IDF e a função cosseno.
3. Escolher 3 consultas e reportar o ranking de cada uma.


## Bloco 1: Reconstruindo o corpus e a matriz TF-IDF

In [ ]:
docs <- c(
  d1 = "recuperacao de informacao ordena documentos por relevancia",
  d2 = "o modelo de espaco vetorial representa documentos como vetores",
  d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
  d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
  d5 = "o indice invertido acelera a busca em muitos documentos",
  d6 = "embeddings capturam a semantica de palavras e documentos",
  d7 = "a avaliacao mede a relevancia dos resultados da busca",
  d8 = "ciencia de dados combina estatistica e programacao"
)

tok <- function(x) unlist(strsplit(tolower(x), "\\s+"))
tokens <- lapply(docs, tok)
vocab <- sort(unique(unlist(tokens)))

tdm <- sapply(tokens, function(t) as.integer(table(factor(t, levels = vocab))))
rownames(tdm) <- vocab
dim(tdm)

N <- ncol(tdm)
df <- rowSums(tdm > 0)
idf <- log(N / df)
w <- tdm * idf
round(w[c("documentos","modelo","de"), ], 2)

[1] 45  8

,d1,d2,d3,d4,d5,d6,d7,d8
documentos,0.69,0.69,0.00,0,0.69,0.69,0,0.00
modelo,0.00,1.39,1.39,0,0.00,0.00,0,0.00
de,0.47,0.47,0.94,0,0.00,0.47,0,0.47


*(Reprodução direta do que já fizemos na Aula 02 em sala: o corpus, a tokenização, o vocabulário e a matriz TF-IDF são os mesmos oito documentos-base usados desde a Aula 01.)*

## Bloco 2: Função de similaridade do cosseno e busca

In [ ]:
cosseno <- function(a, b) sum(a * b) / (sqrt(sum(a^2)) * sqrt(sum(b^2)))

buscar <- function(consulta_txt, w, vocab, idf, top_n = 8) {
  q  <- as.integer(table(factor(tok(consulta_txt), levels = vocab)))
  qw <- q * idf
  scores <- apply(w, 2, function(dvec) cosseno(qw, dvec))
  sort(scores, decreasing = TRUE)[1:top_n]
}

- cosseno(a, b): implementa cos(a,b) = (a.b) / (norma(a) * norma(b)) diretamente da fórmula da Aula 02; mede o ângulo entre dois vetores, ignorando o tamanho de cada um.
- buscar(...): pega uma consulta em texto, tokeniza, monta seu vetor de contagens no mesmo vocabulário do corpus, pondera por IDF (a consulta também é tratada como um "documento"), e calcula o cosseno contra cada coluna (documento) da matriz TF-IDF, devolvendo o ranking completo, do mais para o menos similar.

## Bloco 3: Três consultas e seus rankings

In [ ]:
cat("=== Consulta 1: modelo de recuperacao ===\n")
print(round(buscar("modelo de recuperacao", w, vocab, idf, top_n = 3), 3))

cat("\n=== Consulta 2: busca e avaliacao de relevancia ===\n")
print(round(buscar("busca e avaliacao de relevancia", w, vocab, idf, top_n = 3), 3))

cat("\n=== Consulta 3: ciencia de dados e aprendizado estatistico ===\n")
print(round(buscar("ciencia de dados e aprendizado estatistico", w, vocab, idf, top_n = 3), 3))

=== Consulta 1: modelo de recuperacao ===
   d1    d3    d4 
0.254 0.233 0.215 

=== Consulta 2: busca e avaliacao de relevancia ===
   d7    d1    d5 
0.510 0.167 0.122 

=== Consulta 3: ciencia de dados e aprendizado estatistico ===
   d8    d4    d3 
0.479 0.453 0.065 


**Leitura dos três rankings:**
- **"modelo de recuperacao"** lidera com d1 (fala diretamente de recuperação de informação), seguido por d3 (modelo probabilístico) e d4 (recuperação moderna); todos tematicamente próximos, como esperado.
- **"busca e avaliacao de relevancia"** tem d7 dominando disparado (0.510), porque é o único documento com quase todos os termos da consulta ao mesmo tempo; o mesmo d7 que já tinha concentrado os bits de evidência na Aula 1,5.
- **"ciencia de dados e aprendizado estatistico"** deixa d8 e d4 próximos e bem acima de d3, mostrando que o cosseno diferencia por grau, não só por presença/ausência (diferente da busca booleana da Aula 01).

Essa é a resposta à pergunta central da aula, como medir o quanto um documento combina com a consulta, usando o corpus de brinquedo. A seguir, aplicamos o mesmo método ao corpus real.

---
# Parte 2: Continuando a pesquisa da Baixada Santista

Aplicação da similaridade do cosseno, dando continuidade à pesquisa iniciada na Aula 01 com os verbetes da Wikipedia de Santos, Cubatão e Guarujá, nos mesmos moldes de um trabalho já feito em outra disciplina (o projeto "Similaridade do Cosseno" da Flávia, que usou TF-IDF + similaridade do cosseno para ranquear filmes por proximidade com uma query de preferência do usuário. A lógica é a mesma; aqui o "documento" é o verbete de uma cidade, e a "query" é um interesse de busca (ex: turismo, indústria, porto).

Fonte dos textos: Wikipedia em português (CC BY-SA), mesmos verbetes coletados na Aula 01.

## Bloco 1: Recuperando o corpus das 3 cidades

In [ ]:
library(httr2)

baixar_wiki <- function(titulo) {
  request("https://pt.wikipedia.org/w/api.php") |>
    req_url_query(action = "query", prop = "extracts", explaintext = 1,
                  format = "json", redirects = 1, titles = titulo) |>
    req_perform() |> resp_body_json() |>
    (\(r) r$query$pages[[1]]$extract)()
}

cidades_docs <- c(
  Santos  = baixar_wiki("Santos"),
  Cubatao = baixar_wiki("Cubatao"),
  Guaruja = baixar_wiki("Guaruja")
)
sapply(cidades_docs, nchar)

Santos Cubatao Guaruja 
  54140   16597   19274

Na Aula 01 os tamanhos obtidos foram Santos=54140, Cubatão=16597 e Guarujá=19274 caracteres. Os valores podem variar um pouco se o verbete tiver sido editado desde então.

## Bloco 2: Limpeza, tokenização e TDM (reaproveitando a função da Aula 01)

In [ ]:
stopwords_pt <- c(
  "a","ao","aos","aquela","aquelas","aquele","aqueles","aquilo","as","ate","com","como","da","das",
  "de","dela","delas","dele","deles","depois","do","dos","e","ela","elas","ele","eles","em",
  "entre","era","eram","essa","essas","esse","esses","esta","estamos","estao","estar","estas",
  "estava","estavam","este","estes","eu","foi","fomos","for","fora","foram","ha","isso","isto",
  "ja","lhe","lhes","mais","mas","me","mesmo","meu","meus","minha","minhas","muito","na","nao",
  "nas","nem","no","nos","nossa","nossas","nosso","nossos","num","numa","o","os","ou","para",
  "pela","pelas","pelo","pelos","por","qual","quando","que","quem","se","seja","sejam","sem",
  "sendo","ser","sera","seu","seus","so","somos","sou","sua","suas","tambem","te","tem","temos",
  "tenho","ter","tera","teve","tinha","tinham","tu","tua","tuas","um","uma","voce","voces",
  "vos","sao","sobre","segundo","onde","partir","alem","possui","apos"
)

tokenizar_limpo <- function(texto) {
  texto <- tolower(texto)
  texto <- iconv(texto, to = "ASCII//TRANSLIT")
  texto <- gsub("[^a-z\\s]", " ", texto)
  tk <- unlist(strsplit(texto, "\\s+"))
  tk <- tk[nchar(tk) > 1]
  tk <- tk[!tk %in% stopwords_pt]
  tk
}

tokens_cidades <- lapply(cidades_docs, tokenizar_limpo)
vocab_cidades  <- sort(unique(unlist(tokens_cidades)))

tdm_cidades <- sapply(tokens_cidades, function(tk) {
  as.integer(table(factor(tk, levels = vocab_cidades)))
})
rownames(tdm_cidades) <- vocab_cidades

N_cid  <- ncol(tdm_cidades)
df_cid <- rowSums(tdm_cidades > 0)
idf_cid <- log(N_cid / df_cid)
tfidf_cidades <- tdm_cidades * idf_cid

dim(tfidf_cidades)

[1] 3438    3

**Nota:**  Reaproveita a mesma lista de stopwords e a mesma função de limpeza (minúsculas, remoção de acentos, remoção de pontuação) já validada na Aula 01.

## Bloco 3: Busca por similaridade do cosseno no corpus real

In [ ]:
cosseno <- function(a, b) {
  denom <- sqrt(sum(a^2)) * sqrt(sum(b^2))
  if (denom == 0) return(0)   # documento sem nenhum termo da consulta -> similaridade 0, não NaN
  sum(a * b) / denom
}

buscar_cidade <- function(consulta_txt, tfidf, vocab, idf) {
  termos_q <- tokenizar_limpo(consulta_txt)
  q  <- as.integer(table(factor(termos_q, levels = vocab)))
  qw <- q * idf
  scores <- apply(tfidf, 2, function(v) cosseno(qw, v))
  sort(scores, decreasing = TRUE)
}

cat("=== Consulta: porto industria petroleo ===\n")
print(round(buscar_cidade("porto industria petroleo", tfidf_cidades, vocab_cidades, idf_cid), 4))

cat("\n=== Consulta: praia turismo ===\n")
print(round(buscar_cidade("praia turismo", tfidf_cidades, vocab_cidades, idf_cid), 4))

cat("\n=== Consulta: poluicao industrial saude ===\n")
print(round(buscar_cidade("poluicao industrial saude", tfidf_cidades, vocab_cidades, idf_cid), 4))

=== Consulta: porto industria petroleo ===
 Santos Guaruja Cubatao 
 0.0134  0.0064  0.0000 

=== Consulta: praia turismo ===
Guaruja  Santos Cubatao 
 0.1181  0.0883  0.0000 

=== Consulta: poluicao industrial saude ===
Guaruja Cubatao  Santos 
 0.0197  0.0158  0.0033 


**Nota:** A função buscar_cidade é exatamente a mesma lógica do sistema de recomendação da Flávia: query -> TF-IDF -> cosseno -> ranking; só que sem o filtro de idioma/nota que ela usou para filmes, e com uma consulta curta em vez de uma descrição livre.

## Bloco 4: Verificação com dados já conhecidos da Aula 01 (Execução offline)

In [ ]:
# Sub-vetores reais já calculados na Aula 01 (Bloco 4: Cálculo da Matriz TF-IDF),
# usados aqui só para demonstrar o método sem depender de internet em outro ambiente.
termos_amostra <- c("porto","industria","praia","turismo","petroleo","ferrovia")
tfidf_amostra <- cbind(
  Santos  = c(0.00, 0.41, 7.30, 3.65, 1.10, 1.10),
  Cubatao = c(0,    0,    0,    0,    0,    0   ),
  Guaruja = c(0.00, 0.81, 3.65, 3.65, 0.00, 0.00)
)
rownames(tfidf_amostra) <- termos_amostra

idf_amostra <- c(porto = 0, industria = log(3/2), praia = log(3/2),
                  turismo = log(3/2), petroleo = log(3/1), ferrovia = log(3/1))

buscar_amostra <- function(termos_consulta) {
  q  <- as.integer(rownames(tfidf_amostra) %in% termos_consulta)
  qw <- q * idf_amostra[rownames(tfidf_amostra)]
  scores <- apply(tfidf_amostra, 2, function(v) cosseno(qw, v))
  sort(scores, decreasing = TRUE)
}

cat("=== Consulta: porto + industria + petroleo ===\n")
print(round(buscar_amostra(c("porto","industria","petroleo")), 4))

cat("\n=== Consulta: praia + turismo ===\n")
print(round(buscar_amostra(c("praia","turismo")), 4))

=== Consulta: porto + industria + petroleo ===
 Santos Guaruja Cubatao 
 0.1411  0.0537  0.0000 

=== Consulta: praia + turismo ===
Guaruja  Santos Cubatao 
 0.9879  0.9308  0.0000 


**Leitura:**
- Na consulta **"porto industria petroleo"**, Santos lidera (0.1411), sendo o único dos três verbetes que menciona "petróleo"/"ferrovia" e tem o maior peso relativo em "indústria". Guarujá aparece bem atrás (0.0537) e Cubatão empata em zero.
- Na consulta **"praia turismo"**, a ordem se inverte: Guarujá assume a frente (0.9879), com Santos logo atrás (0.9308). Os dois têm quase a mesma composição relativa de "praia"+"turismo", então o ângulo entre seus vetores e o da consulta é quase idêntico.
- **Cubatão fica em zero nas duas consultas.** E aqui a Aula 1,5 se conecta direto com a Aula 02: nesse recorte de 6 termos, o vetor de Cubatão é literalmente o vetor nulo. O verbete não usa nenhuma dessas 6 palavras específicas (mesmo sendo historicamente a cidade mais industrial das três). Cosseno com o vetor zero não tem direção definida, então a função retorna 0 por convenção. Isso não significa que Cubatão seja "irrelevante" para indústria. Significa que, neste recorte manual de 6 termos, o modelo não tem vocabulário para enxergar a cidade. É o mesmo problema do "motor cego" das rotas (Aula 1,5, Bloco 9): sem contraste de vocabulário, não há distância para medir. Rodando a busca sobre o vocabulário completo (~3428 termos, Bloco 3 acima) esse problema deve diminuir bastante, porque Cubatão certamente tem termos próprios de peso alto (ex: os nomes citados na Aula 01, "sesmaria", "vereador", ainda que esses tragam o problema oposto, de nomes próprios com IDF artificialmente alto).

---
## Conclusão

- A busca booleana da Aula 01 só dizia **se** um termo aparecia; a similaridade do cosseno da Aula 02 diz **o quanto** um documento combina com a consulta, permitindo ranquear.
- O mesmo padrão TF-IDF -> vetor de consulta -> cosseno -> ranking que a Flávia usou para recomendar filmes funciona aqui para recuperar o verbete de cidade mais alinhado com um interesse de busca.
- O limite fica evidente quando o vocabulário é pequeno ou pouco representativo (o caso de Cubatão no recorte de 6 termos): sem contraste, o cosseno não discrimina, o mesmo limite mostrado com as rotas na Aula 1,5.
- Próximo passo natural (Aula 03 em diante): pré-processamento mais cuidadoso do corpus real e, mais adiante, BM25/embeddings para reduzir esse tipo de ponto cego.
